In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.3 Softmax

> 🎯 **Goal:** Turn a list of raw scores into a probability distribution, and understand why softmax is the bridge from a model's opinions to actual predictions.

**Why this matters.** A language model does not directly output "the next word is cat". It outputs a raw score for every possible next token, called a **logit**. Logits can be any number, positive or negative, large or small. Softmax is the function that converts those messy scores into clean probabilities that are all positive and sum to 1. Every time the model picks a next token, softmax happens first.

**The intuition.** Think of betting odds. Suppose three horses have "strength" scores; a higher score should mean a higher chance of winning, but scores alone are not probabilities. Softmax does two things: it makes every score positive by exponentiating (raising e to that power, which turns any number into a positive one and exaggerates the gaps), then it divides each by the total so they add up to 1, like converting raw odds into percentages. A horse twice as strong does not get exactly twice the probability; the exponentiation makes strong favorites pull ahead faster.

**The mechanics.** For a list of logits, softmax is two steps:

1. **Exponentiate:** replace each logit `z` with `exp(z)`. This guarantees positivity and amplifies differences.
2. **Normalize:** divide each exponentiated value by the sum of all of them, so the whole list sums to 1.

That is literally the whole formula: `softmax(z) = exp(z) / sum(exp(z))`. The argument `dim=-1` in `F.softmax` tells PyTorch which dimension to normalize across; `-1` means "the last dimension", which here is the row of three logits. We build it by hand to demystify it, then check it against PyTorch's optimized `F.softmax`.

In [ ]:
from torch.nn import functional as F

logits = torch.tensor([2.0, 1.0, 0.1])
by_hand = logits.exp() / logits.exp().sum()
builtin = F.softmax(logits, dim=-1)
print("by hand: ", by_hand)
print("F.softmax:", builtin)

**What just happened.** Both versions printed the same three numbers, roughly `[0.66, 0.24, 0.10]`, and they add up to 1. The largest logit (`2.0`) got the largest probability, and the gaps got stretched: a logit gap of 1.0 became a probability ratio of nearly 3 to 1. That is a valid distribution over the next token, and it matches PyTorch exactly, which confirms our by-hand formula is the real thing.

⚠️ **Common pitfalls**
- Forgetting to normalize. If you only exponentiate and skip the divide-by-the-sum step, your numbers are positive but do not sum to 1, so they are not probabilities. Sampling from them will misbehave.
- Overflow on large logits. `exp(1000)` is astronomically large and overflows to infinity, which then produces `nan` (not-a-number) when divided. Real implementations like `F.softmax` quietly subtract the maximum logit first (which does not change the result) to stay numerically safe. Prefer the built-in for exactly this reason.

🏋️ **Try it yourself.** Multiply `logits` by `10` before the softmax, then by `0.1`, and rerun each time. Scaling up makes the distribution sharp and confident (almost all probability on the top logit); scaling down makes it flat and uncertain (closer to equal thirds). You just discovered **temperature**, the knob that controls how adventurous a model's sampling is, which returns in later units.

In [ ]:
assert torch.allclose(by_hand, builtin)
assert abs(builtin.sum().item() - 1.0) < 1e-6